
# Marketing Intelligence Dashboard
## Inteligencia de Marketing y Segmentación de Clientes

**Curso:** ADM-3083 — Herramientas y Visualización  
**Caso:** Opción C — Campañas de Marketing  
**Equipo:** Moreno · Albán · Rivas · Arteaga  

---

### Pregunta de negocio central
> *¿Cuál es el perfil del cliente de alto valor y cuál es el canal más efectivo para contactarlo?*



## Tabla de contenidos

| Sección | Contenido |
|---|---|
| **1. Preparación de datos** | Carga, limpieza y creación de variables |
| **2. Paleta corporativa** | Definición de colores del dashboard |
| **3. Acto 1 — Perfil demográfico** | Gráficos 1 y 2: educación y estado civil |
| **4. Acto 2 — Composición familiar** | Gráficos 3 y 4: hijos e ingreso |
| **5. Acto 3 — Canal y campañas** | Gráficos 5 y 6: canal de compra y respuesta |
| **6. Conclusiones y referencias** | Resumen ejecutivo y fuentes |

---

## Parte 1 — Preparación de Datos

Esta sección carga el dataset original, realiza la exploración inicial, ejecuta la limpieza necesaria y construye las variables derivadas que serán utilizadas en todos los gráficos del dashboard.

**Decisiones de limpieza:**
- Se eliminan los 24 registros con `Income` nulo, ya que esta variable es central para el análisis de segmentación y no puede imputarse sin sesgar los resultados.
- Se unifican categorías minoritarias de `Marital_Status` (`YOLO`, `Alone`, `Absurd`) bajo `Single`, ya que representan menos del 0.4% de los datos y no aportan valor analítico.
- Se crean 4 columnas derivadas que facilitan el análisis multidimensional.

In [ ]:
# ─── Librerías base ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from datetime import datetime

print("Librerías base importadas.")

In [ ]:
# ─── Carga del dataset (ruta relativa — compatible con Streamlit Cloud) ───────
df = pd.read_csv("marketing_campaign.csv", sep=",")
df.head()

In [ ]:
# ─── Exploración inicial ──────────────────────────────────────────────────────
print("Dimensiones del dataset:")
print(df.shape)

print("\nColumnas:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos por columna:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# ─── Instalación de librerías de visualización ────────────────────────────────
%pip install plotly pandas statsmodels --quiet
import plotly.express as px
import plotly.graph_objects as go

# ─── Limpieza del dataset ─────────────────────────────────────────────────────

# 1. Eliminar nulos en Income
n_antes = len(df)
df = df.dropna(subset=["Income"])
n_despues = len(df)
print(f"Registros eliminados por Income nulo: {n_antes - n_despues}")

# 2. Unificar categorías minoritarias de estado civil
df["Marital_Status"] = df["Marital_Status"].replace(
    {"YOLO": "Single", "Alone": "Single", "Absurd": "Single"}
)
print(f"Estados civiles válidos: {sorted(df['Marital_Status'].unique())}")

# ─── Creación de columnas derivadas ──────────────────────────────────────────

# Age: edad del cliente calculada al año de análisis
df["Age"] = 2024 - df["Year_Birth"]

# TotalKids: total de dependientes en el hogar (niños + adolescentes)
df["TotalKids"] = df["Kidhome"] + df["Teenhome"]

# TotalSpend: gasto total del cliente en todas las categorías de producto
df["TotalSpend"] = (
    df["MntWines"] + df["MntFruits"] + df["MntMeatProducts"]
    + df["MntFishProducts"] + df["MntSweetProducts"] + df["MntGoldProds"]
)

# TotalCampaigns: número total de campañas aceptadas por el cliente
df["TotalCampaigns"] = (
    df["AcceptedCmp1"] + df["AcceptedCmp2"] + df["AcceptedCmp3"]
    + df["AcceptedCmp4"] + df["AcceptedCmp5"] + df["Response"]
)

# ─── Verificación de resultado ────────────────────────────────────────────────
print(f"\n Dataset limpio: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"\nResumen de columnas creadas:")
print(f"  Age            → rango: {df['Age'].min()}–{df['Age'].max()} años")
print(f"  TotalKids      → rango: {df['TotalKids'].min()}–{df['TotalKids'].max()} dependientes")
print(f"  TotalSpend     → promedio: ${df['TotalSpend'].mean():.0f}")
print(f"  TotalCampaigns → promedio: {df['TotalCampaigns'].mean():.2f} campañas aceptadas")

In [ ]:
# ─── Vista previa de columnas relevantes para el análisis ────────────────────
cols_relevantes = ["Education", "Marital_Status", "Income", "TotalKids",
                   "MntWines", "MntGoldProds", "TotalSpend", "Age"]
df[cols_relevantes].head(8)

In [ ]:
# ─── Estadísticas descriptivas de variables de gasto ─────────────────────────
df[["MntWines", "MntGoldProds", "TotalSpend", "Income"]].describe().round(1)

---
## Parte 2 — Paleta Corporativa
### Integrante 3

Se define la identidad visual del dashboard antes de construir cualquier gráfico.  
La paleta sigue los principios de **Storytelling with Data**: cada color tiene un significado semántico claro, no es decorativo.

| Color | Uso en el dashboard |
|---|---|
| **Violeta vino** | Gráficos de `MntWines` — elemento protagonista del análisis |
| **Dorado** | `MntGoldProds` — productos exclusivos / alto valor |
| **Coral** | Alertas, segmentos de bajo consumo |
| **Verde** | Respuesta positiva a campañas |
| **Grises** | Contexto, ejes, elementos secundarios |

In [ ]:
# ─── Paleta corporativa — compartida por todo el equipo ──────────────────────
PALETTE = {
    "vino":        "#7B2D8B",   # violeta vino  → gráficos de vinos / elemento principal
    "dorado":      "#C9962B",   # dorado        → Gold Products / destacados
    "coral":       "#E05C4B",   # coral         → alertas / segmento bajo consumo
    "verde":       "#3A8C6E",   # verde         → respuesta positiva a campañas
    "gris_claro":  "#D6D6D6",   # gris claro    → elementos neutros / escala baja
    "gris_oscuro": "#4A4A4A",   # gris oscuro   → texto secundario / ejes
    "fondo":       "#FAFAF8",   # blanco cálido → fondo de TODOS los gráficos
}

# Secuencia para gráficos con múltiples categorías
COLOR_SEQ = ["#7B2D8B", "#C9962B", "#3A8C6E", "#E05C4B", "#5B8DB8"]

print("Paleta corporativa definida:")
for nombre, hex_val in PALETTE.items():
    print(f"  {nombre:12s} → {hex_val}")

In [ ]:
# ─── Previsualización de la paleta como swatches ──────────────────────────────
fig_palette = go.Figure()
for nombre, color in PALETTE.items():
    fig_palette.add_trace(go.Bar(
        x=[nombre],
        y=[1],
        marker_color=color,
        name=nombre,
        text=color,
        textposition="inside",
        textfont=dict(color="white" if nombre != "gris_claro" else "#333", size=13),
        showlegend=False,
    ))

fig_palette.update_layout(
    title="Paleta Corporativa — Marketing Intelligence Dashboard",
    plot_bgcolor=PALETTE["fondo"],
    paper_bgcolor=PALETTE["fondo"],
    xaxis=dict(showgrid=False),
    yaxis=dict(visible=False),
    barmode="group",
    height=250,
    margin=dict(t=50, b=20),
)
fig_palette.show()

---
## Parte 3 — Acto 1: ¿Quién es el cliente de alto valor?
### Integrante 3

El primer acto establece el **perfil demográfico** del cliente que más gasta en productos premium.  
Analizamos dos variables: el nivel educativo y el estado civil, para determinar cuál de las dos diferencia mejor el comportamiento de consumo.

**Hipótesis a validar:** *Los clientes con mayor formación académica y sin pareja estable gastan más en vinos y productos exclusivos.*

### Gráfico 1 — *"A mayor nivel educativo, el gasto en vinos se multiplica por 56x"*

**Variable analizada:** `MntWines` promedio por nivel de `Education`  
**Tipo de gráfico:** Barras horizontales con escala de color secuencial  
**Por qué horizontal:** Facilita la lectura de etiquetas de categorías largas y refuerza visualmente la magnitud de las diferencias.

In [ ]:
# ─── Agregación: promedio de MntWines por nivel educativo ─────────────────────
orden_edu = ["Basic", "2n Cycle", "Graduation", "Master", "PhD"]

agg1 = (
    df.groupby("Education")["MntWines"]
    .mean()
    .reindex(orden_edu)
    .dropna()
    .reset_index()
)
agg1.columns = ["Educación", "Gasto Promedio en Vinos ($)"]

print("Datos agregados:")
print(agg1.to_string(index=False))

In [ ]:
# ─── Gráfico 1: Barras horizontales ──────────────────────────────────────────
fig1 = px.bar(
    agg1,
    x="Gasto Promedio en Vinos ($)",
    y="Educación",
    orientation="h",
    title="A mayor nivel educativo, el gasto en vinos se multiplica por 56x",
    color="Gasto Promedio en Vinos ($)",
    color_continuous_scale=[PALETTE["gris_claro"], PALETTE["vino"]],
    text="Gasto Promedio en Vinos ($)",
)

fig1.update_traces(
    texttemplate="$%{text:.0f}",
    textposition="outside",
)

fig1.update_layout(
    plot_bgcolor=PALETTE["fondo"],
    paper_bgcolor=PALETTE["fondo"],
    font=dict(color=PALETTE["gris_oscuro"], size=13),
    coloraxis_showscale=False,
    xaxis=dict(showgrid=False, visible=False),
    yaxis=dict(showgrid=False, title=""),
    margin=dict(l=10, r=100, t=60, b=20),
    height=330,
)

fig1.show()

print("\nInsight:")
print("  Los clientes PhD gastan en promedio $407 en vinos.")
print("  Los clientes Basic gastan apenas $7 — una diferencia de 56x.")
print("  El nivel educativo es el predictor demográfico más fuerte del consumo premium.")

### Gráfico 2 — *"Clientes sin pareja lideran el gasto en productos premium"*

**Variables analizadas:** `MntWines` y `MntGoldProds` promedio por `Marital_Status`  
**Tipo de gráfico:** Barras agrupadas — permite comparar dos categorías de gasto simultáneamente  
**Insight esperado:** Validar si el estado civil refuerza o contradice el patrón observado en educación.

In [ ]:
# ─── Agregación: promedio de Vinos y Gold por estado civil ────────────────────
agg2 = (
    df.groupby("Marital_Status")[["MntWines", "MntGoldProds"]]
    .mean()
    .reset_index()
    .sort_values("MntWines", ascending=False)
)
agg2.columns = ["Estado Civil", "Gasto en Vinos ($)", "Gasto en Gold ($)"]

print("Datos agregados (ordenados por gasto en vinos):")
print(agg2.to_string(index=False))

In [ ]:
# ─── Gráfico 2: Barras agrupadas ──────────────────────────────────────────────
fig2 = px.bar(
    agg2,
    x="Estado Civil",
    y=["Gasto en Vinos ($)", "Gasto en Gold ($)"],
    title="Clientes sin pareja lideran el gasto en productos premium",
    barmode="group",
    color_discrete_map={
        "Gasto en Vinos ($)": PALETTE["vino"],
        "Gasto en Gold ($)":  PALETTE["dorado"],
    },
)

fig2.update_layout(
    plot_bgcolor=PALETTE["fondo"],
    paper_bgcolor=PALETTE["fondo"],
    font=dict(color=PALETTE["gris_oscuro"], size=13),
    legend_title_text="Categoría de gasto",
    xaxis=dict(title="", showgrid=False),
    yaxis=dict(title="Gasto Promedio ($)", showgrid=True, gridcolor="#EBEBEB"),
    margin=dict(t=60, b=20),
    height=380,
)

fig2.show()

print("\nInsight:")
print("  Viudos: ~$728 promedio en vinos — el más alto de todos los segmentos.")
print("  Solteros y divorciados: ~$610–612.")
print("  Casados: $591 — el grupo más numeroso pero el de menor gasto promedio.")
print("  Recomendación: segmentar campañas diferenciadas según estado civil.")

---
## Parte 4 — Acto 2: ¿La familia frena el consumo premium?
### Integrante 3

El Acto 1 estableció que el perfil educativo y el estado civil predicen el gasto.  
El Acto 2 profundiza en la **composición familiar**: ¿tiene hijos el cliente? ¿Cuántos?  
Y cruza esta variable con el ingreso para entender si la restricción de presupuesto o la priorización del gasto es lo que diferencia a los segmentos.

**Hipótesis a validar:** *La presencia de hijos en el hogar reduce el gasto en productos premium, independientemente del nivel de ingreso.*

### Gráfico 3 — *"Cada hijo adicional reduce el gasto en vinos hasta en un 71%"*

**Variable analizada:** `MntWines` promedio según `TotalKids`  
**Tipo de gráfico:** Línea con área rellena — ideal para mostrar tendencias continuas y la magnitud de la caída acumulada.

In [ ]:
# ─── Agregación: promedio de MntWines por número total de hijos ───────────────
agg3 = (
    df.groupby("TotalKids")["MntWines"]
    .mean()
    .reset_index()
)
agg3.columns = ["Hijos en el Hogar", "Gasto Promedio en Vinos ($)"]

# Calcular reducción porcentual respecto al segmento sin hijos
base = agg3.loc[agg3["Hijos en el Hogar"] == 0, "Gasto Promedio en Vinos ($)"].values[0]
print("Datos agregados:")
for _, row in agg3.iterrows():
    reduccion = (1 - row["Gasto Promedio en Vinos ($)"] / base) * 100
    print(f"  {int(row['Hijos en el Hogar'])} hijos → ${row['Gasto Promedio en Vinos ($)']:.0f}  ({-reduccion:.0f}% vs sin hijos)")

In [ ]:
# ─── Gráfico 3: Línea con área rellena ───────────────────────────────────────
fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x=agg3["Hijos en el Hogar"],
    y=agg3["Gasto Promedio en Vinos ($)"],
    mode="lines+markers+text",
    line=dict(color=PALETTE["vino"], width=3),
    marker=dict(size=11, color=PALETTE["vino"]),
    fill="tozeroy",
    fillcolor="rgba(123, 45, 139, 0.12)",
    text=[f"${v:.0f}" for v in agg3["Gasto Promedio en Vinos ($)"]],
    textposition="top center",
    textfont=dict(size=13, color=PALETTE["vino"]),
    name="Gasto en Vinos",
))

fig3.update_layout(
    title="Cada hijo adicional reduce el gasto en vinos hasta en un 71%",
    plot_bgcolor=PALETTE["fondo"],
    paper_bgcolor=PALETTE["fondo"],
    font=dict(color=PALETTE["gris_oscuro"], size=13),
    xaxis=dict(
        title="Número de hijos en el hogar",
        tickvals=[0, 1, 2, 3],
        ticktext=["0 hijos", "1 hijo", "2 hijos", "3 hijos"],
        showgrid=False,
    ),
    yaxis=dict(
        title="Gasto Promedio en Vinos ($)",
        showgrid=True,
        gridcolor="#EBEBEB",
    ),
    margin=dict(t=60, b=20),
    height=360,
    showlegend=False,
)

fig3.show()

print("\nInsight:")
print("  Sin hijos → $488 | 1 hijo → $269 (-45%) | 2 hijos → $142 (-71%)")
print("  La presencia de hijos es el factor inhibidor más fuerte del consumo premium.")
print("  El segmento sin hijos debe ser el objetivo prioritario de las campañas.")

### Gráfico 4 — *"Ingreso alto + sin hijos = el cliente de alto valor ideal"*

**Variables analizadas:** `Income` vs `TotalSpend`, segmentado por composición familiar  
**Tipo de gráfico:** Scatter con línea de tendencia (OLS) — muestra correlación y permite comparar clusters simultáneamente.  
**Decisión técnica:** Se excluye el top 1% de ingresos para evitar que outliers distorsionen la escala.

In [ ]:
# ─── Preparación: excluir outliers extremos de ingreso (top 1%) ───────────────
percentil_99 = df["Income"].quantile(0.99)
df_plot = df[df["Income"] < percentil_99].copy()

# Crear segmento familiar legible
df_plot["Composición Familiar"] = df_plot["TotalKids"].apply(
    lambda x: "Sin hijos" if x == 0 else ("1 hijo" if x == 1 else "2+ hijos")
)

print(f"Filas tras filtro de outliers: {len(df_plot)} (de {len(df)} originales)")
print(f"\nDistribución por composición familiar:")
print(df_plot["Composición Familiar"].value_counts().to_string())

In [ ]:
# ─── Gráfico 4: Scatter con trendline ────────────────────────────────────────
color_map = {
    "Sin hijos": PALETTE["vino"],
    "1 hijo":    PALETTE["dorado"],
    "2+ hijos":  PALETTE["coral"],
}

fig4 = px.scatter(
    df_plot,
    x="Income",
    y="TotalSpend",
    color="Composición Familiar",
    color_discrete_map=color_map,
    title="Ingreso alto + sin hijos = el cliente de alto valor ideal",
    opacity=0.55,
    trendline="ols",
    trendline_scope="overall",
    trendline_color_override=PALETTE["gris_oscuro"],
    labels={
        "Income":               "Ingreso Anual ($)",
        "TotalSpend":           "Gasto Total en Productos ($)",
        "Composición Familiar": "Composición Familiar",
    },
)

fig4.update_layout(
    plot_bgcolor=PALETTE["fondo"],
    paper_bgcolor=PALETTE["fondo"],
    font=dict(color=PALETTE["gris_oscuro"], size=13),
    xaxis=dict(showgrid=False, tickformat="$,.0f"),
    yaxis=dict(showgrid=True, gridcolor="#EBEBEB"),
    margin=dict(t=60, b=20),
    height=430,
)

fig4.show()

print("\n💡 Insight:")
print("  A igual nivel de ingreso, los clientes sin hijos siempre aparecen")
print("  más arriba en el eje Y que los de 1 o 2+ hijos.")
print("  El ingreso no basta: la composición familiar amplifica o frena el potencial de gasto.")
print("  Perfil ideal → ingreso alto + sin hijos en el hogar.")

---
## Parte 5 — Acto 3: ¿Por dónde llegamos al cliente ideal?
### Integrante 4 — José Moreno

Los dos actos anteriores establecieron *quién* es el cliente de alto valor:  
educado, sin hijos en casa, con ingresos altos y alto gasto en vinos y productos exclusivos.

Ahora la pregunta estratégica cambia: **¿cómo lo contactamos?**

Este acto analiza dos dimensiones clave para la toma de decisiones del equipo de marketing:
1. **El canal de compra:** ¿dónde prefiere comprar este cliente? ¿Web, tienda física, catálogo?
2. **La respuesta a campañas:** ¿qué segmento tiene mayor probabilidad de aceptar una oferta?

Los hallazgos de este acto son los más **accionables** del dashboard, ya que permiten asignar presupuesto al canal correcto y al segmento correcto.

---

In [ ]:
# ─── Preparación general del Acto 3 ──────────────────────────────────────────
# Columna auxiliar: perfil familiar simplificado (binario)
df["TieneHijos"] = df["TotalKids"].apply(lambda x: "Sin hijos" if x == 0 else "Con hijos")

print("Columna auxiliar 'TieneHijos' creada.")
print(df["TieneHijos"].value_counts().to_string())

### Gráfico 5 — *"El catálogo es el canal clave para el cliente sin hijos, con 2.7x más uso que sus pares con familia"*

**Variables analizadas:** Promedio de `NumWebPurchases`, `NumStorePurchases`, `NumCatalogPurchases`, `NumDealsPurchases`, segmentado por `TieneHijos`  
**Tipo de gráfico:** Barras agrupadas — permite comparar cuatro canales a la vez entre dos segmentos  
**Por qué importa:** El canal no es neutro. Saber dónde compra cada segmento permite personalizar el punto de contacto y optimizar el presupuesto.

In [ ]:
# ─── Preparación de datos: Gráfico 5 ─────────────────────────────────────────
canales_cols = {
    "NumWebPurchases"     : "Web",
    "NumStorePurchases"   : "Tienda Física",
    "NumCatalogPurchases" : "Catálogo",
    "NumDealsPurchases"   : "Ofertas / Deals",
}

canal_df = (
    df.groupby("TieneHijos")[list(canales_cols.keys())]
    .mean()
    .reset_index()
    .rename(columns=canales_cols)
    .melt(id_vars="TieneHijos", var_name="Canal", value_name="Promedio de Compras")
)

print("Promedio de compras por canal y perfil familiar:")
print(canal_df.pivot(index="Canal", columns="TieneHijos", values="Promedio de Compras").round(1).to_string())

In [ ]:
# ─── Gráfico 5: Barras agrupadas por canal y perfil familiar ──────────────────
fig5 = px.bar(
    canal_df,
    x="Canal",
    y="Promedio de Compras",
    color="TieneHijos",
    barmode="group",
    title="El catálogo es el canal clave para el cliente sin hijos, con 2.7x más uso que sus pares con familia",
    labels={"Promedio de Compras": "Promedio de transacciones", "TieneHijos": "Perfil familiar"},
    color_discrete_map={"Sin hijos": PALETTE["vino"], "Con hijos": PALETTE["gris_claro"]},
    text_auto=".1f",
)

fig5.update_layout(
    plot_bgcolor=PALETTE["fondo"],
    paper_bgcolor=PALETTE["fondo"],
    font=dict(color=PALETTE["gris_oscuro"], size=13),
    legend_title_text="Perfil familiar",
    xaxis_title=None,
    yaxis_title="Promedio de transacciones",
    bargap=0.25,
    margin=dict(t=70, b=20),
    height=400,
)

fig5.update_traces(textposition="outside")
fig5.show()

print("\nInsight:")
print("  Canal catálogo: Sin hijos → 4.8 compras | Con hijos → 1.8 compras (diferencia: 2.7x)")
print("  Canal tienda física: Sin hijos → 7.3 | Con hijos → 5.2")
print("  Canal ofertas/deals: Con hijos → 2.8 | Sin hijos → 1.1 (único canal donde Con hijos supera)")
print("  Conclusión: el cliente sin hijos explora más (catálogo, web).")
print("  El cliente con hijos responde mejor a ofertas directas — prioriza el ahorro inmediato.")

#### Narrativa del Gráfico 5 — Para el dashboard (`st.info`)

> **Hallazgo principal:** La tienda física es el canal más utilizado en ambos segmentos, pero los clientes **sin hijos** superan a los que tienen hijos en el canal de **catálogo** (4.8 vs 1.8 compras en promedio) y en **web**. Esto sugiere que el cliente de alto valor tiene mayor disposición para explorar y descubrir productos por cuenta propia.

> **Hallazgo secundario:** El único canal donde los clientes **con hijos** superan a los sin hijos es en **Ofertas/Deals** (2.8 vs 1.1). Este segmento responde al incentivo económico inmediato, no a la exploración de catálogos.

> **Implicación estratégica:** El presupuesto de marketing de catálogo y digital debe concentrarse en el segmento sin hijos. Para clientes con hijos, los descuentos y promociones directas son el canal de entrada más eficiente.

### Gráfico 6 — *"Los clientes con PhD y sin hijos son 2.4x más propensos a aceptar una campaña que el promedio"*

**Variable analizada:** Tasa de respuesta (`Response = 1`) segmentada por `Education` y `TieneHijos`  
**Tipo de gráfico:** Barras agrupadas con línea de referencia — el promedio general sirve de ancla para evaluar qué segmentos están sobre y bajo la media  
**Por qué importa:** No todos los segmentos responden igual a una campaña. Identificar quién acepta más permite concentrar el esfuerzo donde el retorno es mayor.

In [ ]:
# ─── Preparación de datos: Gráfico 6 ─────────────────────────────────────────
orden_educacion = ["Basic", "2n Cycle", "Graduation", "Master", "PhD"]

resp_df = (
    df.groupby(["Education", "TieneHijos"])["Response"]
    .mean()
    .reset_index()
)
resp_df["Tasa de Respuesta (%)"] = (resp_df["Response"] * 100).round(1)
resp_df["Education"] = pd.Categorical(resp_df["Education"], categories=orden_educacion, ordered=True)
resp_df = resp_df.sort_values("Education")

tasa_promedio = df["Response"].mean() * 100

print(f"Tasa de respuesta promedio general: {tasa_promedio:.1f}%")
print("\nDetalle por segmento:")
print(resp_df[["Education", "TieneHijos", "Tasa de Respuesta (%)"]].to_string(index=False))

In [ ]:
# ─── Gráfico 6: Barras agrupadas con línea de referencia ─────────────────────
fig6 = px.bar(
    resp_df,
    x="Education",
    y="Tasa de Respuesta (%)",
    color="TieneHijos",
    barmode="group",
    title="Los clientes con PhD y sin hijos son 2.4x más propensos a aceptar una campaña que el promedio",
    labels={"Education": "Nivel de educación", "TieneHijos": "Perfil familiar"},
    color_discrete_map={"Sin hijos": PALETTE["verde"], "Con hijos": PALETTE["gris_claro"]},
    text_auto=".1f",
)

# Línea de referencia: tasa promedio general
fig6.add_hline(
    y=tasa_promedio,
    line_dash="dot",
    line_color=PALETTE["coral"],
    annotation_text=f"Promedio general: {tasa_promedio:.1f}%",
    annotation_position="top left",
    annotation_font_color=PALETTE["coral"],
)

fig6.update_layout(
    plot_bgcolor=PALETTE["fondo"],
    paper_bgcolor=PALETTE["fondo"],
    font=dict(color=PALETTE["gris_oscuro"], size=13),
    legend_title_text="Perfil familiar",
    xaxis_title="Nivel educativo",
    yaxis=dict(
        title="% que aceptó la campaña",
        ticksuffix="%",
        showgrid=True,
        gridcolor="#EBEBEB",
    ),
    bargap=0.25,
    margin=dict(t=70, b=20),
    height=410,
)

fig6.update_traces(textposition="outside")
fig6.show()

print(f"\n Insight:")
print(f"  PhD + Sin hijos: 36.1% de tasa de respuesta — 2.4x el promedio general ({tasa_promedio:.1f}%)")
print(f"  PhD + Con hijos: 15.2% — prácticamente igual al promedio")
print(f"  Basic (ambos segmentos): por debajo del 5% — retorno marginal")
print(f"  Conclusión: la educación y la familia son variables que se potencian mutuamente.")

#### Narrativa del Gráfico 6 — Para el dashboard (`st.info`)

> **Hallazgo principal:** La tasa de respuesta escala con el nivel educativo en ambos segmentos, pero la brecha es dramática en el nivel PhD: los clientes **PhD sin hijos** responden al **36.1%** — más del doble del promedio general (15%). Los PhD con hijos, en cambio, responden al 15.2%, prácticamente igual que el promedio.

> **Hallazgo secundario:** Los clientes con educación básica (`Basic`) tienen tasas por debajo del 5% en ambos segmentos. Dirigir campañas a este grupo implica un alto costo de oportunidad con retorno marginal.

> **Implicación estratégica:** La próxima campaña de alto valor debe concentrar sus impactos en clientes con nivel **Master o PhD sin hijos**. Este segmento no solo responde más, sino que — como demostraron los Actos 1 y 2 — también gasta más en vinos y productos exclusivos: es el mismo cliente en todos los ejes.

---
## Parte 6 — Conclusiones y Recomendaciones
### Equipo

### Resumen ejecutivo del análisis

A lo largo de los tres actos, el dashboard respondió la pregunta de negocio central con hallazgos concretos y accionables:

| Acto | Pregunta | Hallazgo clave |
|---|---|---|
| **Acto 1** | ¿Quién gasta más? | El nivel educativo es el predictor más fuerte: los PhD gastan 56x más en vinos que los clientes de educación básica |
| **Acto 2** | ¿La familia importa? | Cada hijo adicional reduce el gasto en vinos hasta un 71%. A igual ingreso, los clientes sin hijos siempre gastan más |
| **Acto 3** | ¿Cómo los contactamos? | El catálogo es el canal clave para el cliente sin hijos (2.7x más uso). Los PhD sin hijos responden a campañas a una tasa del 36.1% — el doble del promedio |

---

### Perfil del cliente de alto valor

El análisis converge en un perfil claro:
- **Educación:** Master o PhD
- **Composición familiar:** Sin hijos en el hogar
- **Ingreso:** Segmento medio-alto a alto
- **Canal preferido:** Catálogo y tienda física
- **Probabilidad de respuesta a campaña:** 2–2.4x el promedio

### Recomendaciones estratégicas

1. **Segmentar** la base de clientes para campañas de vinos y gold hacia clientes con educación Master/PhD y sin hijos — es donde el retorno es máximo.
2. **Activar el canal de catálogo** para este segmento: es su canal de exploración natural y donde la competencia por atención es menor que en digital.
3. **No invertir** presupuesto de productos premium en clientes con educación básica o con 2+ hijos — el historial de respuesta no justifica el costo.
4. **Para clientes con hijos**, diseñar una campaña separada basada en **ofertas y descuentos directos** (Deals), que es el único canal donde este segmento supera al otro.

---

## Referencias

Knaflic, C. N. (2015). *Storytelling with data: A data visualization guide for business professionals*. Wiley.

Rodsaldanha. (s.f.). *Marketing campaign* [Conjunto de datos]. Kaggle. https://www.kaggle.com/datasets/rodsaldanha/arketing-campaign

McKinney, W. (2022). *Python for data analysis* (3.ª ed.). O'Reilly Media.

Plotly Technologies Inc. (2015). *Collaborative data science*. Plotly. https://plotly.com

Streamlit Inc. (2019). *Streamlit — The fastest way to build data apps*. https://streamlit.io

---
*Proyecto Fin de Semestre — ADM-3083 Herramientas y Visualización*  
*Equipo: Moreno · Albán · Rivas · Arteaga*